In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os, json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix, recall_score
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

DRIVE_DIR   = "/content/drive/MyDrive/Corneal_v2"
RESULTS_DIR = "/content/drive/MyDrive/CNN_results_v2"
CSV_FILE = os.path.join(DRIVE_DIR, "Final_Features_MultiLabel_updated.csv")
SEV_FILE = os.path.join(DRIVE_DIR, "Severity_Labels_PerDisease_updated.xlsx")
SEV_DIR  = os.path.join(RESULTS_DIR, "severity_densenet121")           # Severity_DenseNet121_v2
OUT_DIR  = os.path.join(RESULTS_DIR, "combined_final")
os.makedirs(OUT_DIR, exist_ok=True)

# disease predictions: full comparison notebook (preferred) or the fusion-only notebook
DIS_CANDIDATES = [os.path.join(RESULTS_DIR, "final_densenet121", "test_predictions_all.csv"),          # Fusion_DenseNet121_Retrain_v2
                  os.path.join(RESULTS_DIR, "final_fusion_densenet121", "test_predictions_fusion.csv")]  # fusion-only, BACKBONE=densenet121
DIS_FILE = next((f for f in DIS_CANDIDATES if os.path.exists(f)), None)
assert DIS_FILE, "Disease predictions not found - run Fusion_DenseNet121_Retrain_v2 to the end first"
for f in (CSV_FILE, SEV_FILE, os.path.join(SEV_DIR, "summary_severity.json")):
    assert os.path.exists(f), f"Missing: {f}"

DIS_COLS = ["FECD", "Guttata", "Folds", "PolyPleo", "LowCD"]

dis      = pd.read_csv(DIS_FILE)
sev_pred = pd.read_csv(os.path.join(SEV_DIR, "test_predictions_severity.csv"))
sev_oof  = pd.read_csv(os.path.join(SEV_DIR, "oof_predictions.csv"))
sev_sum  = json.load(open(os.path.join(SEV_DIR, "summary_severity.json")))
labels   = pd.read_excel(SEV_FILE, sheet_name="Severity_Labels", keep_default_na=False, na_values=[""])
feat     = pd.read_csv(CSV_FILE)

SEV_HEADS = sev_sum["severity_heads"]            # e.g. {"FECDgroup": [...], "Folds": [...], "Guttata": [...]}
SEV_THR   = sev_sum["severity_thresholds"]
HEAD_NAME = {"FECDgroup": "FECD group", "Folds": "Folds", "Guttata": "Guttata"}
HAS_NUM   = all(f"P_Num_{c}" in dis.columns for c in DIS_COLS)

print("Disease predictions :", DIS_FILE, dis.shape)
print("Severity predictions:", sev_pred.shape)
print("Severity outputs    :", list(SEV_HEADS))
print("Severity thresholds :", {k: round(v, 3) for k, v in SEV_THR.items()})

Disease predictions : /content/drive/MyDrive/CNN_results_v2/final_densenet121/test_predictions_all.csv (305, 38)
Severity predictions: (305, 21)
Severity outputs    : ['FECDgroup', 'Folds', 'Guttata']
Severity thresholds : {'FECDgroup': 0.57, 'Folds': 0.491, 'Guttata': 0.434}


Merges disease predictions, severity probabilities, true severity labels, and numerical biomarkers using Tissue_ID

In [3]:
t = (dis[["Tissue_ID", "Combination"] + DIS_COLS + [f"P_Fus_{c}" for c in DIS_COLS] + [f"Pred_Fus_{c}" for c in DIS_COLS]]
     .merge(sev_pred[["Tissue_ID", "Label_Origin"] + [f"PSev_{h}" for h in SEV_HEADS]], on="Tissue_ID")
     .merge(labels[["Tissue_ID"] + [f"{h}_Sev" for h in SEV_HEADS] + [c for c in ["Guttata_Sev", "Overall_Sev"]
                                                                       if c not in [f"{h}_Sev" for h in SEV_HEADS]]], on="Tissue_ID")
     .merge(feat[["Tissue_ID", "ECD_CD_cells_per_mm2", "CV", "HEX_percent", "Donor_Age"]], on="Tissue_ID", how="left"))
print(f"Test corneas combined: {len(t)} | disease file {len(dis)} | severity file {len(sev_pred)}")
assert len(t) == len(dis) == len(sev_pred), "The two notebooks used different test rows - re-run both with the same CSV/images"
t.head(3)

Test corneas combined: 305 | disease file 305 | severity file 305


,Tissue_ID,Combination,FECD,Guttata,Folds,PolyPleo,LowCD,P_Fus_FECD,P_Fus_Guttata,P_Fus_Folds,P_Fus_PolyPleo,P_Fus_LowCD,Pred_Fus_FECD,Pred_Fus_Guttata,Pred_Fus_Folds,Pred_Fus_PolyPleo,Pred_Fus_LowCD,Label_Origin,PSev_FECDgroup,PSev_Folds,PSev_Guttata,FECDgroup_Sev,Folds_Sev,Guttata_Sev,Overall_Sev,ECD_CD_cells_per_mm2,CV,HEX_percent,Donor_Age
0,NEB-2020-09-12663 R,Normal,0,0,0,0,0,0.003929,0.318359,0.871094,0.189453,0.000207,0,0,1,0,0,Original,0.899414,0.740234,0.810547,None,None,None,None,2740.0,41.0,56.0,72.0
1,NEB-2020-09-12667 R,Normal,0,0,0,0,0,0.002728,0.142334,0.827637,0.119385,0.000083,0,0,1,0,0,Original,0.537109,0.284180,0.508301,None,None,None,None,2833.0,34.0,61.0,69.0
2,NEB-2020-09-12673 L,FECD + PolyPleo + LowCD,1,0,0,1,1,0.604492,0.056213,0.782715,0.787109,0.917480,1,0,1,1,1,Original,0.547852,0.732422,0.688477,Mild,None,None,Mild,2016.0,35.0,48.0,59.0


Calculates thresholds for disease and severity predictions while assigning a higher cost to missed disease cases. This allows the system to prioritize sensitivity when missing a disease or moderate/severe case is considered more harmful than a false alarm.

In [4]:
# ===== Cost-sensitive thresholds (COST_FN = 1 gives exactly Youden's J) =====
from sklearn.metrics import roc_curve
USE_COST_THRESHOLDS = True        # False = keep Youden everywhere
COST_FN_DIS = {"FECD": 3.0, "Guttata": 2.5, "LowCD": 2.0, "PolyPleo": 1.5, "Folds": 1.5}   # how much worse a missed case is
COST_FN_SEV = {"FECDgroup": 2.0, "Guttata": 2.0, "Folds": 1.5}                            # Mod/Sev called Mild

def cost_threshold(y, p, c_fn, c_fp=1.0):
    fpr, tpr, thr = roc_curve(y, p)
    return float(np.clip(thr[np.argmin(c_fn * (1 - tpr) + c_fp * fpr)], 0.01, 0.99))

def row(out, method, split, y, p, thr):
    yhat = (p >= thr).astype(int)
    return {"Output": out, "Method": method, "Split": split, "Threshold": round(thr, 3),
            "Sensitivity": recall_score(y, yhat, zero_division=0),
            "Specificity": recall_score(y, yhat, pos_label=0, zero_division=0),
            "Missed (FN)": int(((yhat == 0) & (y == 1)).sum()), "False alarms (FP)": int(((yhat == 1) & (y == 0)).sum())}

# disease: out-of-fold predictions + Youden cut-offs saved by notebook 06
dis_oof = pd.read_csv(os.path.join(RESULTS_DIR, "fusion_densenet121", "oof_predictions.csv")).drop_duplicates("Tissue_ID")
DIS_YOUDEN = json.load(open(os.path.join(RESULTS_DIR, "final_densenet121", "summary_final.json")))["thresholds"]["Image + numbers"]
DIS_COST = {c: cost_threshold(dis_oof[c].values, dis_oof[f"P_{c}"].values, COST_FN_DIS[c]) for c in DIS_COLS}

# severity: out-of-fold predictions saved by notebook 08 (only corneas with a known Mild/Mod/Sev grade)
so = sev_oof.merge(labels[["Tissue_ID"] + [f"{h}_Sev" for h in SEV_HEADS]], on="Tissue_ID")
def sev_xy(frame, h):
    m = frame[f"{h}_Sev"].isin(["Mild", "Moderate", "Severe"])
    return frame.loc[m, f"{h}_Sev"].isin(["Moderate", "Severe"]).astype(int).values, frame.loc[m, f"PSev_{h}"].values
SEV_COST = {h: cost_threshold(*sev_xy(so, h), COST_FN_SEV[h]) for h in SEV_HEADS}

rows = []
for c in DIS_COLS:
    for nm, thr in (("Youden J", DIS_YOUDEN), ("Cost-sensitive", DIS_COST)):
        rows.append(row(c, nm, "Train CV", dis_oof[c].values, dis_oof[f"P_{c}"].values, thr[c]))
        rows.append(row(c, nm, "Test", t[c].values, t[f"P_Fus_{c}"].values, thr[c]))
for h in SEV_HEADS:
    for nm, thr in (("Youden J", SEV_THR), ("Cost-sensitive", SEV_COST)):
        rows.append(row(f"Severity {h}", nm, "Train CV", *sev_xy(so, h), thr[h]))
        rows.append(row(f"Severity {h}", nm, "Test", *sev_xy(t, h), thr[h]))
cost_cmp = pd.DataFrame(rows).set_index(["Output", "Split", "Method"]).sort_index()
cost_cmp.to_csv(os.path.join(OUT_DIR, "cost_vs_youden.csv"))
print("Youden vs cost-sensitive - choose the costs on 'Train CV' rows, report 'Test' rows once")
display(cost_cmp.round(3))

# sensitivity analysis on train CV: threshold and sensitivity for different cost ratios
sa = pd.DataFrame({f"cost {r}": {c: f"{cost_threshold(dis_oof[c].values, dis_oof[f'P_{c}'].values, r):.3f}" for c in DIS_COLS}
                   for r in [1, 1.5, 2, 3, 5]})
sa.to_csv(os.path.join(OUT_DIR, "cost_sensitivity_analysis.csv"))
print("Disease threshold for each cost ratio (train CV; cost 1 = Youden)"); display(sa)

# apply the chosen cut-offs to the combined table
DIS_THR_USED = DIS_COST if USE_COST_THRESHOLDS else DIS_YOUDEN
for c in DIS_COLS:
    t[f"Pred_Fus_{c}"] = (t[f"P_Fus_{c}"] >= DIS_THR_USED[c]).astype(int)
print("Disease cut-offs used:", {k: round(v, 3) for k, v in DIS_THR_USED.items()},
      "| method:", "cost-sensitive" if USE_COST_THRESHOLDS else "Youden")

Youden vs cost-sensitive - choose the costs on 'Train CV' rows, report 'Test' rows once


Threshold  Sensitivity  Specificity  Missed (FN)  False alarms (FP)
Output             Split    Method                                                                             
FECD               Test     Cost-sensitive      0.235        1.000        0.747            0                 70
                            Youden J            0.235        1.000        0.747            0                 70
                   Train CV Cost-sensitive      0.235        0.956        0.746            5                281
                            Youden J            0.235        0.956        0.746            5                281
Folds              Test     Cost-sensitive      0.453        0.864        0.563           21                 66
                            Youden J            0.487        0.851        0.603           23                 60
                   Train CV Cost-sensitive      0.453        0.846        0.589           95                248
                            Youden J            0.487        0.816        0.627          114                225
Guttata            Test     Cost-sensitive      0.183        0.921        0.573            3                114
                            Youden J            0.442        0.763        0.805            9                 52
                   Train CV Cost-sensitive      0.183        0.904        0.502           15                531
                            Youden J            0.442        0.724        0.801           43                212
LowCD              Test     Cost-sensitive      0.260        1.000        0.943            0                 13
                            Youden J            0.466        1.000        0.952            0                 11
                   Train CV Cost-sensitive      0.260        0.978        0.937            7                 57
                            Youden J            0.466        0.965        0.953           11                 43
PolyPleo           Test     Cost-sensitive      0.440        0.754        0.843           17                 37
                            Youden J            0.440        0.754        0.843           17                 37
                   Train CV Cost-sensitive      0.440        0.781        0.845           64                144
                            Youden J            0.440        0.781        0.845           64                144
Severity FECDgroup Test     Cost-sensitive      0.334        0.875        0.500            6                 25
                            Youden J            0.570        0.771        0.820           11                  9
                   Train CV Cost-sensitive      0.334        0.881        0.483           22                104
                            Youden J            0.570        0.649        0.801           65                 40
Severity Folds     Test     Cost-sensitive      0.420        0.842        0.698            9                 29
                            Youden J            0.491        0.737        0.781           15                 21
                   Train CV Cost-sensitive      0.420        0.773        0.697           51                118
                            Youden J            0.491        0.711        0.769           65                 90
Severity Guttata   Test     Cost-sensitive      0.343        0.636        0.615            4                 10
                            Youden J            0.434        0.545        0.692            5                  8
                   Train CV Cost-sensitive      0.343        0.800        0.598            8                 45
                            Youden J            0.434        0.700        0.705           12                 33

Disease threshold for each cost ratio (train CV; cost 1 = Youden)


,cost 1,cost 1.5,cost 2,cost 3,cost 5
FECD,0.235,0.235,0.235,0.235,0.049
Guttata,0.442,0.276,0.247,0.183,0.085
Folds,0.487,0.453,0.410,0.201,0.173
PolyPleo,0.440,0.440,0.413,0.174,0.107
LowCD,0.466,0.260,0.260,0.114,0.114


Disease cut-offs used: {'FECD': 0.235, 'Guttata': 0.183, 'Folds': 0.453, 'PolyPleo': 0.44, 'LowCD': 0.26} | method: cost-sensitive


Tests different probability thresholds using cross-validation predictions to identify how well the model detects moderate/severe cases while keeping mild cases correctly classified.

In [5]:
for h in SEV_HEADS:
    tr = sev_oof.merge(labels[["Tissue_ID", f"{h}_Sev"]], on="Tissue_ID")
    tr = tr[tr[f"{h}_Sev"].isin(["Mild", "Moderate", "Severe"])]
    y = tr[f"{h}_Sev"].isin(["Moderate", "Severe"]).astype(int).values
    p = tr[f"PSev_{h}"].values
    n_mild, n_mod = int((y == 0).sum()), int(y.sum())
    rows = []
    for thr in [SEV_THR[h], 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
        yhat = (p >= thr).astype(int)
        rows.append({"Cut-off": round(thr, 3), "Sensitivity (catch Mod/Sev)": recall_score(y, yhat, zero_division=0),
                     "Specificity (keep Mild as Mild)": recall_score(y, yhat, pos_label=0, zero_division=0),
                     f"Mild wrongly flagged (of {n_mild})": int(((yhat == 1) & (y == 0)).sum()),
                     f"Mod/Sev missed (of {n_mod})": int(((yhat == 0) & (y == 1)).sum())})
    print(f"\n{HEAD_NAME.get(h, h)} severity cut-off options (training CV, {n_mild} Mild / {n_mod} Mod+Sev) - first row = automatic (Youden):")
    display(pd.DataFrame(rows).round(3))


FECD group severity cut-off options (training CV, 201 Mild / 185 Mod+Sev) - first row = automatic (Youden):


,Cut-off,Sensitivity (catch Mod/Sev),Specificity (keep Mild as Mild),Mild wrongly flagged (of 201),Mod/Sev missed (of 185)
0,0.57,0.649,0.801,40,65
1,0.30,0.892,0.438,113,20
2,0.35,0.859,0.493,102,26
3,0.40,0.805,0.557,89,36
4,0.45,0.773,0.647,71,42
5,0.50,0.708,0.701,60,54
6,0.55,0.654,0.771,46,64
7,0.60,0.589,0.811,38,76



Folds severity cut-off options (training CV, 390 Mild / 225 Mod+Sev) - first row = automatic (Youden):


,Cut-off,Sensitivity (catch Mod/Sev),Specificity (keep Mild as Mild),Mild wrongly flagged (of 390),Mod/Sev missed (of 225)
0,0.491,0.711,0.769,90,65
1,0.300,0.880,0.513,190,27
2,0.350,0.840,0.559,172,36
3,0.400,0.782,0.656,134,49
4,0.450,0.747,0.731,105,57
5,0.500,0.680,0.782,85,72
6,0.550,0.613,0.826,68,87
7,0.600,0.573,0.874,49,96



Guttata severity cut-off options (training CV, 112 Mild / 40 Mod+Sev) - first row = automatic (Youden):


,Cut-off,Sensitivity (catch Mod/Sev),Specificity (keep Mild as Mild),Mild wrongly flagged (of 112),Mod/Sev missed (of 40)
0,0.434,0.700,0.705,33,12
1,0.300,0.800,0.536,52,8
2,0.350,0.775,0.607,44,9
3,0.400,0.725,0.679,36,11
4,0.450,0.625,0.705,33,15
5,0.500,0.550,0.750,28,18
6,0.550,0.425,0.812,21,23
7,0.600,0.350,0.893,12,26


Chooses either the cost-sensitive thresholds or the automatically calculated thresholds. It also selects whether FECD severity should come from the trained model or a cell-density rule.

In [6]:
THR = dict(SEV_COST) if USE_COST_THRESHOLDS else dict(SEV_THR)            # automatic cut-offs from Severity_DenseNet121_v2
# THR["Folds"] = 0.40            # <- uncomment / edit after looking at the tables above
FECD_SEV_METHOD = "model"        # "model" = severity notebook output | "cd_rule" = cell density < 1600 -> Mod/Sev
print({k: round(v, 3) for k, v in THR.items()}, "| FECD method:", FECD_SEV_METHOD)

{'FECDgroup': 0.334, 'Folds': 0.42, 'Guttata': 0.343} | FECD method: model


Combines the detected diseases with their predicted severity levels. It then assigns an overall severity of None, Mild, or Mod/Sev, using the most severe predicted condition.

In [7]:
ORD = {"None": 0, "Mild": 1, "Mod/Sev": 2}

def head_severity(r, h):
    if h == "FECDgroup" and FECD_SEV_METHOD == "cd_rule":
        return "Mod/Sev" if r["ECD_CD_cells_per_mm2"] < 1600 else "Mild"
    return "Mod/Sev" if r[f"PSev_{h}"] >= THR[h] else "Mild"

def cornea_report(r):
    found = [c for c in DIS_COLS if r[f"Pred_Fus_{c}"] == 1]          # diseases from the fusion notebook
    sevs = [(HEAD_NAME.get(h, h), head_severity(r, h)) for h, ds in SEV_HEADS.items() if any(c in found for c in ds)]
    if "Guttata" in found and not any("Guttata" in ds for ds in SEV_HEADS.values()):
        sevs.append(("Guttata", "Mild"))                               # only if no Guttata severity output
    overall = max([s for _, s in sevs], key=lambda s: ORD[s]) if sevs else "None"
    return pd.Series({"Predicted diseases": " + ".join(found) if found else "Normal",
                      "Predicted severity": ", ".join(f"{d}: {s}" for d, s in sevs) if sevs else "-",
                      "Predicted overall": overall})

to3 = lambda s: s.astype(str).replace({"Moderate": "Mod/Sev", "Severe": "Mod/Sev", "?": "Mod/Sev", "nan": "None"})
rep = pd.concat([t, t.apply(cornea_report, axis=1)], axis=1)
rep["True overall"] = to3(rep["Overall_Sev"])

L = ["None", "Mild", "Mod/Sev"]
cm = pd.DataFrame(confusion_matrix(rep["True overall"], rep["Predicted overall"], labels=L),
                  index=[f"True {l}" for l in L], columns=[f"Pred {l}" for l in L])
print("FINAL end-to-end overall severity (test): fusion diseases + severity model")
display(cm)
agree = (rep["True overall"] == rep["Predicted overall"]).mean()
sev_cm = pd.DataFrame(sev_sum.get("overall_confusion", {}))
sev_alone = np.trace(sev_cm.values) / sev_cm.values.sum() if sev_cm.size else np.nan
print(f"Exact agreement: {agree:.1%}   (severity notebook alone: {sev_alone:.1%})")
print(f"Mod/Sev corneas called Normal: {int(cm.loc['True Mod/Sev', 'Pred None'])} of {int(cm.loc['True Mod/Sev'].sum())}")

FINAL end-to-end overall severity (test): fusion diseases + severity model


,Pred None,Pred Mild,Pred Mod/Sev
True None,37,48,16
True Mild,5,63,65
True Mod/Sev,0,11,60


Exact agreement: 52.5%   (severity notebook alone: 65.9%)
Mod/Sev corneas called Normal: 0 of 71


Evaluates severity predictions for cases where the disease-screening model detected the corresponding disease.

In [8]:
rows = []
for h, ds in SEV_HEADS.items():
    true_col = f"{h}_Sev"
    m = rep[true_col].isin(["Mild", "Moderate", "Severe"]) & rep[[f"Pred_Fus_{c}" for c in ds]].max(axis=1).eq(1)
    yt = rep.loc[m, true_col].isin(["Moderate", "Severe"]).astype(int)
    yp = rep.loc[m].apply(lambda r: head_severity(r, h), axis=1).eq("Mod/Sev").astype(int) if m.any() else pd.Series(dtype=int)
    rows.append({"Disease": HEAD_NAME.get(h, h), "Truly present": int(rep[true_col].isin(["Mild", "Moderate", "Severe"]).sum()),
                 "Detected": int(m.sum()), "Mild / Mod+Sev (detected)": f"{int((yt == 0).sum())} / {int(yt.sum())}",
                 "Severity correct": f"{(yt == yp).mean():.1%}" if len(yt) else "-",
                 "Sensitivity": recall_score(yt, yp, zero_division=0) if len(yt) else np.nan,
                 "Specificity": recall_score(yt, yp, pos_label=0, zero_division=0) if len(yt) else np.nan})
per_dis = pd.DataFrame(rows).set_index("Disease")
per_dis.to_csv(os.path.join(OUT_DIR, "severity_on_detected_cases.csv"))
print("Severity per disease on DETECTED cases (test)"); display(per_dis.round(3))

Severity per disease on DETECTED cases (test)


,Truly present,Detected,Mild / Mod+Sev (detected),Severity correct,Sensitivity,Specificity
Disease,,,,,,
FECD group,98,83,43 / 40,68.7%,0.900,0.488
Folds,153,133,80 / 53,74.4%,0.868,0.662
Guttata,37,34,24 / 10,61.8%,0.600,0.625


Displays example cornea reports alongside their true overall severity. It also identifies moderate/severe cases that the system incorrectly classified as less severe.

In [9]:
show = ["Tissue_ID", "Combination", "Predicted diseases", "Predicted severity", "Predicted overall", "True overall"]
print("Example final cornea reports (test):")
display(rep[show].sample(min(15, len(rep)), random_state=3).reset_index(drop=True))
print("\nMistakes where a Mod/Sev cornea was under-called:")
display(rep.loc[(rep["True overall"] == "Mod/Sev") & (rep["Predicted overall"] != "Mod/Sev"), show])

Example final cornea reports (test):


,Tissue_ID,Combination,Predicted diseases,Predicted severity,Predicted overall,True overall
0,NEB-2024-08-16268 R,Folds,Guttata + Folds,"Folds: Mild, Guttata: Mild",Mild,Mild
1,NEB-2020-02-12074 L,PolyPleo,Guttata + Folds + PolyPleo + LowCD,"FECD group: Mod/Sev, Folds: Mod/Sev, Guttata: ...",Mod/Sev,Mod/Sev
2,NEB-2020-10-12719 R,Folds,Guttata + Folds,"Folds: Mild, Guttata: Mod/Sev",Mod/Sev,Mild
3,NEB-2023-07-15254 R,Normal,Normal,-,None,None
4,NEB-2023-03-14873 R,Normal,Normal,-,None,None
5,NEB-2020-10-12743 L,Normal,Folds,Folds: Mild,Mild,None
6,NEB-2023-02-14732 R,Normal,Normal,-,None,None
7,NEB-2022-11-14454 L,Folds + LowCD,FECD + Guttata + Folds + PolyPleo + LowCD,"FECD group: Mod/Sev, Folds: Mod/Sev, Guttata: ...",Mod/Sev,Mod/Sev
8,NEB-2025-11-17642 L,Normal,Normal,-,None,None
9,NEB-2023-02-14717 R,Folds,Guttata + Folds,"Folds: Mild, Guttata: Mild",Mild,Mild



Mistakes where a Mod/Sev cornea was under-called:


,Tissue_ID,Combination,Predicted diseases,Predicted severity,Predicted overall,True overall
72,NEB-2023-04-14936 R,Folds,Folds,Folds: Mild,Mild,Mod/Sev
158,NEB-2024-08-16275 L,Guttata + Folds,Guttata,Guttata: Mild,Mild,Mod/Sev
192,NEB-2025-10-17595 R,Guttata,Guttata,Guttata: Mild,Mild,Mod/Sev
210,NEB-2020-06-12374 L,FECD + Guttata + PolyPleo + LowCD,FECD + Guttata + PolyPleo + LowCD,"FECD group: Mild, Guttata: Mild",Mild,Mod/Sev
240,NEB-2024-09-16332 L,PolyPleo + LowCD,FECD + PolyPleo + LowCD,FECD group: Mild,Mild,Mod/Sev
248,NEB-2025-08-17405 R,Guttata + Folds + PolyPleo + LowCD,FECD + PolyPleo + LowCD,FECD group: Mild,Mild,Mod/Sev
270,NEB-2020-01-11956 L,Guttata + Folds,Guttata,Guttata: Mild,Mild,Mod/Sev
274,NEB-2020-01-11964 L,Folds,Folds,Folds: Mild,Mild,Mod/Sev
285,NEB-2020-02-12019 L,Guttata + Folds + PolyPleo,Guttata,Guttata: Mild,Mild,Mod/Sev
298,NEB-2020-12-12894 R,Folds,Folds,Folds: Mild,Mild,Mod/Sev


In [10]:
keep = (["Tissue_ID", "Label_Origin", "Combination"] + DIS_COLS
        + [c for c in [f"{h}_Sev" for h in SEV_HEADS] + ["Guttata_Sev"] if c in rep.columns][:len(SEV_HEADS) + 1]
        + ["True overall"] + [f"P_Fus_{c}" for c in DIS_COLS] + [f"Pred_Fus_{c}" for c in DIS_COLS]
        + [f"PSev_{h}" for h in SEV_HEADS]
        + ["Predicted diseases", "Predicted severity", "Predicted overall", "ECD_CD_cells_per_mm2", "CV", "HEX_percent", "Donor_Age"])
keep = list(dict.fromkeys(keep))
rep[keep].to_csv(os.path.join(OUT_DIR, "Final_Cornea_Reports_test.csv"), index=False)
cm.to_csv(os.path.join(OUT_DIR, "overall_severity_confusion.csv"))
json.dump({"disease_model": f"DenseNet121 image+numbers fusion ({os.path.basename(DIS_FILE)})",
           "severity_model": "DenseNet121 multi-task (Severity_DenseNet121_v2)", "n_test": len(rep),
           "severity_thresholds": THR, "fecd_sev_method": FECD_SEV_METHOD,
           "disease_thresholds": DIS_THR_USED,
           "threshold_method": "cost-sensitive" if USE_COST_THRESHOLDS else "youden",
           "overall_exact_agreement": round(float(agree), 4)},
          open(os.path.join(OUT_DIR, "combined_summary.json"), "w"), indent=2)
print("Saved to", OUT_DIR); print(os.listdir(OUT_DIR))

Saved to /content/drive/MyDrive/CNN_results_v2/combined_final
['severity_on_detected_cases.csv', 'Final_Cornea_Reports_test.csv', 'overall_severity_confusion.csv', 'combined_summary.json', 'disease_auc_by_label_origin.csv', 'ood_reference_v2.npz', 'cost_vs_youden.csv', 'cost_sensitivity_analysis.csv']


In [11]:
g = dis.merge(sev_pred[["Tissue_ID", "Label_Origin"]], on="Tissue_ID")
def auc_or_nan(y, p):
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan
rows = []
for org in ["All test", "Original", "Model-assisted (2023)"]:
    s = g if org == "All test" else g[g["Label_Origin"] == org]
    for c in DIS_COLS:
        r = {"Group": f"{org} (n={len(s)})", "Disease": c, "Cases": int(s[c].sum()),
             "Image + numbers AUC": auc_or_nan(s[c].values, s[f"P_Fus_{c}"].values)}
        if HAS_NUM: r["Numbers only AUC"] = auc_or_nan(s[c].values, s[f"P_Num_{c}"].values)
        rows.append(r)
res = pd.DataFrame(rows)
print("Disease AUC on the SAME test predictions, split by who made the label")
display(res.pivot(index="Disease", columns="Group", values="Image + numbers AUC").round(3))
if HAS_NUM:
    print("\nNumbers-only model, same split (for comparison)")
    display(res.pivot(index="Disease", columns="Group", values="Numbers only AUC").round(3))
print("\nNumber of diseased corneas per group")
display(res.pivot(index="Disease", columns="Group", values="Cases"))
res.to_csv(os.path.join(OUT_DIR, "disease_auc_by_label_origin.csv"), index=False)

Disease AUC on the SAME test predictions, split by who made the label


Group,All test (n=305),Model-assisted (2023) (n=103),Original (n=202)
Disease,,,
FECD,0.918,0.965,0.872
Folds,0.804,0.843,0.784
Guttata,0.880,0.910,0.872
LowCD,0.985,1.000,0.977
PolyPleo,0.852,0.949,0.801



Numbers-only model, same split (for comparison)


Group,All test (n=305),Model-assisted (2023) (n=103),Original (n=202)
Disease,,,
FECD,0.870,0.940,0.874
Folds,0.611,0.581,0.614
Guttata,0.599,0.688,0.567
LowCD,0.982,1.000,0.975
PolyPleo,0.858,0.877,0.848



Number of diseased corneas per group


Group,All test (n=305),Model-assisted (2023) (n=103),Original (n=202)
Disease,,,
FECD,28,18,10
Folds,154,42,112
Guttata,38,9,29
LowCD,75,24,51
PolyPleo,69,22,47
